# NB-A · 154-Class Data Prep — Hand Landmarks Only

Builds train/test NPZs for the top 154 glosses (>=7 clips).
**Hand landmarks only:** hand_index 0 (left) and 1 (right), all 21 landmarks each.
**No face, no body.** Raw MediaPipe coordinates, no normalization.

## Outputs (saved to OBrown_DIS9300_v2/)
```
ASL_154_train_cif.npz   (n, 126, 40)   CIF + InceptionTime
ASL_154_test_cif.npz    (n, 126, 40)
ASL_154_train_rf.csv    stat features   RF + LR
ASL_154_test_rf.csv
ASL_154_meta.json
```
Results folder: `results/cif_154/`, `results/rf_154/`, `results/lr_154/`, `results/inc_154/`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os, json, time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

PROJECT_DIR = ('/content/drive/MyDrive/Consultant/Colab_Notebooks/'
               'Obrown_Dissertation_NU_25/OBrown_DIS9300_v2')
INPUT_CSV   = ('/content/drive/MyDrive/Consultant/Colab_Notebooks/'
               'Obrown_Dissertation_NU_25/OBrown_DIS9300_v3/master_landmarks_full.csv')
os.makedirs(PROJECT_DIR, exist_ok=True)

# Create result subfolders
for sub in ['results/cif_154/checkpoints','results/rf_154',
            'results/lr_154','results/inc_154']:
    os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)

MIN_CLIPS    = 7       # 154 classes
TEST_FRAC    = 0.25
T_FRAMES     = 40     # resampled frames
RANDOM_STATE = 42

# Output paths
TRAIN_NPZ = os.path.join(PROJECT_DIR, 'ASL_154_train_cif.npz')
TEST_NPZ  = os.path.join(PROJECT_DIR, 'ASL_154_test_cif.npz')
TRAIN_CSV = os.path.join(PROJECT_DIR, 'ASL_154_train_rf.csv')
TEST_CSV  = os.path.join(PROJECT_DIR, 'ASL_154_test_rf.csv')
META_PATH = os.path.join(PROJECT_DIR, 'ASL_154_meta.json')
LE_PATH   = os.path.join(PROJECT_DIR, 'ASL_154_label_encoder.pkl')

print(f'Project dir : {PROJECT_DIR}')
print(f'Input CSV   : {INPUT_CSV}')
print(f'MIN_CLIPS   : {MIN_CLIPS}  (154 classes)')
print(f'T_FRAMES    : {T_FRAMES}')
print(f'TEST_FRAC   : {TEST_FRAC}')


Project dir : /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2
Input CSV   : /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v3/master_landmarks_full.csv
MIN_CLIPS   : 7  (154 classes)
T_FRAMES    : 40
TEST_FRAC   : 0.25


In [ ]:
# ── Load CSV, filter to 154 classes, hand landmarks only ─────────────────
print('Loading CSV...')
df = pd.read_csv(INPUT_CSV)
df = df[~df[['x','y','z']].isna().all(axis=1)].copy()
print(f'Total rows: {len(df):,}')

# Class filter: >= MIN_CLIPS
clip_counts  = df.groupby('gloss')['filename'].nunique()
keep_glosses = clip_counts[clip_counts >= MIN_CLIPS].index
df = df[df['gloss'].isin(keep_glosses)].copy()
N_CLASSES = len(keep_glosses)
print(f'After class filter: {N_CLASSES} glosses | {df.filename.nunique()} clips')
assert N_CLASSES == 154, f'Expected 154 got {N_CLASSES}'

# Hand landmarks only: hand_index 0 (left) and 1 (right), ALL 21 landmarks
df = df[df['hand_index'].isin([0, 1])].copy()
print(f'After hand filter : {len(df):,} rows')

# Channel names: 2 hands x 21 landmarks x 3 coords = 126 channels
combos = (df[['hand_index','landmark']]
          .drop_duplicates().sort_values(['hand_index','landmark']))
CHANNEL_NAMES = [
    f"{int(r.hand_index)}_{int(r.landmark)}_{c}"
    for _, r in combos.iterrows() for c in ['x','y','z']
]
N_CH = len(CHANNEL_NAMES)
print(f'Channels: {N_CH}  (2 hands x 21 landmarks x 3 coords)')
assert N_CH == 126, f'Expected 126 channels got {N_CH}'

# Group by clip
groups = {cid: grp for cid, grp in df.groupby('filename', sort=False)}
print(f'Clips: {len(groups):,}')


Loading CSV...
Total rows: 14,078,977
After class filter: 154 glosses | 1239 clips
After hand filter : 1,514,835 rows
Channels: 126  (2 hands x 21 landmarks x 3 coords)
Clips: 1,235


In [ ]:
# ── Pivot each clip to (126, T_FRAMES) tensor ─────────────────────────────
def pivot_clip(group):
    group = group.sort_values('frame')
    long  = group.melt(
        id_vars=['frame','hand_index','landmark'],
        value_vars=['x','y','z'], var_name='coord', value_name='val'
    )
    long['ch'] = (
        long.hand_index.astype(int).astype(str) + '_' +
        long.landmark.astype(int).astype(str)   + '_' + long.coord
    )
    pivot = long.pivot_table(
        index='frame', columns='ch', values='val', aggfunc='mean'
    ).reindex(columns=CHANNEL_NAMES)
    arr = pivot.to_numpy(dtype=np.float32).T  # (126, T_obs)
    # Forward-fill then zero-fill missing values
    for c in range(arr.shape[0]):
        row = arr[c]
        if np.isnan(row).any():
            if np.isnan(row).all():
                arr[c] = 0.0
            else:
                idx = np.where(~np.isnan(row), np.arange(len(row)), 0)
                np.maximum.accumulate(idx, out=idx)
                row = row[idx]
                if np.isnan(row).any():
                    idx2 = np.where(~np.isnan(row), np.arange(len(row)), len(row)-1)
                    np.minimum.accumulate(idx2[::-1], out=idx2[::-1])
                    row = row[idx2]
                arr[c] = row
    # Resample to T_FRAMES
    T = arr.shape[1]
    idx = np.linspace(0, T-1, T_FRAMES).round().astype(int)
    return np.nan_to_num(arr[:, idx], nan=0.0).astype(np.float32)

print(f'Processing {len(groups):,} clips...')
tensors, labels, names = [], [], []
clip_ids = sorted(groups.keys())
for i, clip_id in enumerate(clip_ids):
    grp   = groups[clip_id]
    gloss = grp['gloss'].iloc[0]
    tensors.append(pivot_clip(grp))
    labels.append(gloss)
    names.append(clip_id)
    if (i+1) % 500 == 0 or (i+1) == len(clip_ids):
        print(f'  {i+1:,}/{len(clip_ids):,}')

X_all = np.stack(tensors).astype(np.float32)  # (n, 126, 40)
y_all = np.array(labels)
n_all = np.array(names)
print(f'X_all: {X_all.shape}  NaN={np.isnan(X_all).sum()}')


Processing 1,235 clips...
  500/1,235
  1,000/1,235
  1,235/1,235
X_all: (1235, 126, 40)  NaN=0


In [ ]:
# ── 75/25 stratified video-level split ────────────────────────────────────
indices = np.arange(len(y_all))
tr_idx, te_idx = train_test_split(
    indices, test_size=TEST_FRAC, stratify=y_all, random_state=RANDOM_STATE)
tr_idx = np.array(tr_idx, dtype=int)
te_idx = np.array(te_idx, dtype=int)

# Guarantee no class is test-only
test_only = set(y_all[te_idx]) - set(y_all[tr_idx])
if test_only:
    move   = np.array([i for i in te_idx if y_all[i] in test_only])
    te_idx = np.array([i for i in te_idx if y_all[i] not in test_only])
    tr_idx = np.concatenate([tr_idx, move])
    tr_idx = np.array(tr_idx, dtype=int)
    te_idx = np.array(te_idx, dtype=int)

overlap = set(n_all[tr_idx]) & set(n_all[te_idx])
print(f'Train: {len(tr_idx):,} clips | {len(set(y_all[tr_idx]))} classes')
print(f'Test : {len(te_idx):,} clips | {len(set(y_all[te_idx]))} classes')
print(f'Video overlap (must be 0): {len(overlap)}')

# Label encoder
le = LabelEncoder()
le.fit(np.unique(y_all))
joblib.dump(le, LE_PATH)
print(f'Label encoder: {len(le.classes_)} classes saved')


Train: 926 clips | 154 classes
Test : 309 clips | 154 classes
Video overlap (must be 0): 0
Label encoder: 154 classes saved


In [ ]:
# ── Save NPZs ──────────────────────────────────────────────────────────────
for tag, idx in [('train', tr_idx), ('test', te_idx)]:
    X = X_all[idx]; y = le.transform(y_all[idx]); nm = n_all[idx]
    path = TRAIN_NPZ if tag=='train' else TEST_NPZ
    np.savez_compressed(path, X=X, y=y, video_names=nm,
                        channel_names=np.array(CHANNEL_NAMES),
                        classes=le.classes_)
    mb = os.path.getsize(path)/1024**2
    print(f'{tag} NPZ: {X.shape}  classes={len(set(y))}  {mb:.1f}MB  -> {path}')

# ── Save RF/LR stat feature CSVs ──────────────────────────────────────────
STAT_NAMES = ['mean','std','min','max','median']
feat_names = [f'{ch}_{s}' for ch in CHANNEL_NAMES for s in STAT_NAMES]

def make_stat_df(X, y_enc, nm):
    stats = np.concatenate([
        X.mean(axis=2), X.std(axis=2), X.min(axis=2),
        X.max(axis=2), np.median(X, axis=2)
    ], axis=1)  # (n, 126*5=630)
    stats = stats.reshape(len(X), 5, 126).transpose(0,2,1).reshape(len(X), 630)
    df = pd.DataFrame(stats, columns=feat_names)
    df.insert(0, 'filename', nm)
    df.insert(1, 'gloss',    le.inverse_transform(y_enc))
    return df

d_tr = np.load(TRAIN_NPZ, allow_pickle=True)
d_te = np.load(TEST_NPZ,  allow_pickle=True)
make_stat_df(d_tr['X'], d_tr['y'], d_tr['video_names']).to_csv(TRAIN_CSV, index=False)
make_stat_df(d_te['X'], d_te['y'], d_te['video_names']).to_csv(TEST_CSV,  index=False)
print(f'Train CSV: {TRAIN_CSV}')
print(f'Test  CSV: {TEST_CSV}')

# ── Metadata ──────────────────────────────────────────────────────────────
meta = {
    'n_classes':154,'min_clips':MIN_CLIPS,'T_frames':T_FRAMES,
    'test_frac':TEST_FRAC,'n_channels':N_CH,
    'landmarks':'hand_only_hi0_hi1_all_21',
    'normalization':'none_raw_mediapipe',
    'n_train':int(len(tr_idx)),'n_test':int(len(te_idx)),
    'channel_names':CHANNEL_NAMES,
    'stat_features':feat_names,
}
with open(META_PATH,'w') as f: json.dump(meta,f,indent=2)
print(f'Meta: {META_PATH}')
print('\n== NB-A complete. Next: run NB-B (CIF), NB-C (RF/LR), NB-D (InceptionTime) ==')


train NPZ: (926, 126, 40)  classes=154  10.3MB  -> /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/ASL_154_train_cif.npz
test NPZ: (309, 126, 40)  classes=154  3.5MB  -> /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/ASL_154_test_cif.npz
Train CSV: /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/ASL_154_train_rf.csv
Test  CSV: /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/ASL_154_test_rf.csv
Meta: /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/ASL_154_meta.json

== NB-A complete. Next: run NB-B (CIF), NB-C (RF/LR), NB-D (InceptionTime) ==
